<a href="https://colab.research.google.com/github/chiaraco16/COVID-19-analysis/blob/main/COVID_L5_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 L5 – ETL Classico: dal Reconciled DB allo Star Schema

**Progetto:** COVID-19 Trend Analysis — Chiara Costantino (277081)

Questo notebook implementa l'**ETL Phase 2** del progetto, guidato dal `schema.json` prodotto da `COVID_L4_Data_Modeling.ipynb`.
- **`ETLAuditLog`** — traccia ogni step
- **`ETLPipeline`** — orchestrazione Extract → Transform → Load driven by `schema.json`
- **Surrogate keys** generate via `range(1, N+1)`
- **`Dim_Time` generata** dal range di date del Reconciled DB
- **Export CSV** per Tableau Public

### Flusso del notebook
1. Caricamento `schema.json` (contratto da L4)
2. Extract: tre tabelle del Reconciled DB (`Locations`, `Epidemic_Trend`, `Prevention_Measures`)
3. Transform: `Dim_Time`, `Dim_Location_Context`, `Fact_Covid_Trend` con SK + audit
4. Load: export CSV + audit log + verifica integrità
5. Sample OLAP query come sanity check


## 0. Drive e configurazione percorsi

In [15]:
# Monto Google Drive per leggere i file del progetto
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os

BASE_PATH = '/content/drive/MyDrive/Covid-19 Analysis/'

# Input: schema.json prodotto da L4 + tabelle pulite da L3
SCHEMA_PATH        = BASE_PATH + 'schema.json'
EPIDEMIC_CLEAN     = BASE_PATH + 'L3_epidemic_trend_clean.csv'
PREVENTION_CLEAN   = BASE_PATH + 'L3_prevention_measures_clean.csv'
LOCATIONS_CLEAN    = BASE_PATH + 'L3_locations_clean.csv'

# Output del notebook
DW_DIR             = BASE_PATH + 'dw/'
OUT_DIM_TIME       = DW_DIR + 'Dim_Time.csv'
OUT_DIM_LOC        = DW_DIR + 'Dim_Location_Context.csv'
OUT_FACT           = DW_DIR + 'Fact_Covid_Trend.csv'
OUT_AUDIT          = DW_DIR + 'etl_log.csv'
OUT_SCHEMA_UPDATED = BASE_PATH + 'schema.json'

os.makedirs(DW_DIR, exist_ok=True)

## 1. Librerie

In [17]:
# Librerie del notebook
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime
from pathlib import Path

pd.set_option('display.max_columns', 50)

## 2. Caricamento `schema.json`



In [18]:
# Leggo schema.json: descrive fatto, dimensioni e misure del DW
with open(SCHEMA_PATH, 'r', encoding='utf-8') as f:
    schema = json.load(f)

print('Schema caricato')
print(f'   Generato da: {schema.get("source", "n/a")}')
print(f'   Progetto:    {schema.get("project", "n/a")}')
print()

# Estrazione delle parti rilevanti per l'ETL
ss          = schema['star_schema']
fact_spec   = ss['fact_table']
dim_specs   = ss['dimension_tables']
measures    = schema['measures']   # con additivity + aggregation

Schema caricato
   Generato da: COVID_L4_Data_Modeling.ipynb
   Progetto:    COVID-19 Trend Analysis — Data Warehouse



## 3. `ETLAuditLog`

Ogni step registra `kind`, `rows_in`, `rows_out`, `duration`, `status`.

In [19]:
# Classe per registrare ogni passo dell'ETL
class ETLAuditLog:
    """Log auditabile di ogni step ETL."""

    def __init__(self):
        self.entries = []

    def step(self, name, kind, rows_in, rows_out,
             duration, status='ok', note=''):
        self.entries.append({
            'timestamp': datetime.now().isoformat(timespec='seconds'),
            'step':      name,
            'kind':      kind,           # extract / transform / load
            'rows_in':   rows_in,
            'rows_out':  rows_out,
            'duration':  round(duration, 3),
            'status':    status,
            'note':      note
        })

    def to_dataframe(self):
        return pd.DataFrame(self.entries)

    def print_summary(self):
        df = self.to_dataframe()
        if df.empty:
            print('(audit log vuoto)')
            return
        print('ETL AUDIT LOG')
        print('=' * 95)
        for _, r in df.iterrows():
            icon = '[OK]' if r['status'] == 'ok' else '[FAIL]'
            print(f"  {icon} {r['step']:<30} {r['kind']:<10} "
                  f"in={r['rows_in']:>7,}  out={r['rows_out']:>7,}  "
                  f"{r['duration']:>6.2f}s  {r['note']}")
        print('=' * 95)
        print(f"Totale step: {len(df)} | Durata complessiva: {df['duration'].sum():.2f}s")

audit = ETLAuditLog()
print('ETLAuditLog inizializzato')

ETLAuditLog inizializzato


## 4. `ETLPipeline`

La classe orchestra Extract / Transform / Load. Ogni metodo:
1. Marca il tempo iniziale
2. Esegue il lavoro
3. Registra lo step nell'audit log



In [20]:
# Pipeline ETL guidata dallo schema:
#   Extract (leggo i CSV) -> Transform (costruisco dimensioni e fatto) -> Load (salvo i CSV)
import numpy as np
class ETLPipeline:
    """Pipeline ETL guidata da schema.json."""

    def __init__(self, schema, audit):
        self.schema = schema
        self.audit  = audit
        self.src    = {}   # Reconciled tables (extract output)
        self.dw     = {}   # DW tables (transform output)

    # ------------------------------ EXTRACT ------------------------------
    def extract(self, name, path):
        t0 = time.perf_counter()
        df = pd.read_csv(path)
        # Coerci date se presente
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
        self.src[name] = df
        self.audit.step(name, 'extract', len(df), len(df),
                        time.perf_counter() - t0,
                        note=f'da {Path(path).name}')
        return df

    # ------------------------------ TRANSFORM ----------------------------
    def build_dim_time(self):
        """Genera Dim_Time dal range di date del Reconciled DB.

        Pattern del corso: la dim_date NON si estrae, si GENERA dal range
        delle date osservate. Aggiunge SK e attributi gerarchici.
        """
        t0 = time.perf_counter()
        ep = self.src['Epidemic_Trend']
        all_dates = pd.date_range(
            start=ep['date'].min(),
            end=ep['date'].max(),
            freq='D'
        )
        start = pd.Timestamp('2020-01-01')
        df = pd.DataFrame({
            'time_sk':          range(1, len(all_dates) + 1),
            'date':             all_dates,
            'month':            all_dates.to_period('M').astype(str),
            'quarter':          all_dates.to_period('Q').astype(str),
            'year':             all_dates.year,
            'day_of_week':      all_dates.dayofweek,
            'is_weekend':       all_dates.dayofweek >= 5,
            'days_since_start': (all_dates - start).days
        })
        df['etl_loaded_at'] = datetime.now().isoformat(timespec='seconds')

        self.dw['Dim_Time'] = df
        self.audit.step('Dim_Time', 'transform', len(ep), len(df),
                        time.perf_counter() - t0,
                        note=f'generata da {ep["date"].nunique()} date distinte')
        return df

    def build_dim_location_context(self):
        """Costruisce Dim_Location_Context dal Reconciled DB.

        Pattern: drop_duplicates su natural key, SK = range(1, N+1),
        livelli gerarchici letti da schema.json.

        UNITA' DI MISURA (documentate per coerenza Tableau):
          population                   -> persone (conteggio assoluto)
          population_density           -> persone per km^2
          gdp_per_capita               -> USD a parita' di potere d'acquisto
          median_age                   -> anni
          hospital_beds_per_thousand   -> posti letto per 1.000 abitanti
          human_development_index      -> indice 0-1
        """
        t0 = time.perf_counter()
        loc = self.src['Locations']
        spec = self.schema['star_schema']['dimension_tables']['Dim_Location_Context']
        nk = spec['natural_key']   # 'iso_code'

        # Attributi previsti dallo schema (coerenti con schema_dw.sql e report)
        DIM_LOC_COLS = ['iso_code', 'location', 'continent', 'population',
                        'population_density', 'gdp_per_capita', 'median_age',
                        'hospital_beds_per_thousand', 'human_development_index']

        df = (loc
              .dropna(subset=[nk])
              .drop_duplicates(subset=[nk])
              .reset_index(drop=True))

        # Tengo solo gli attributi dello schema (scarto le colonne OWID extra)
        df = df[[c for c in DIM_LOC_COLS if c in df.columns]].copy()

        # Coerenza tipi: i numerici come float, le chiavi/testi come stringa
        num_cols = ['population', 'population_density', 'gdp_per_capita',
                    'median_age', 'hospital_beds_per_thousand',
                    'human_development_index']
        for c in num_cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce')
        for c in ['iso_code', 'location', 'continent']:
            if c in df.columns:
                df[c] = df[c].astype('string')

        # SK
        df.insert(0, spec['surrogate_key'], range(1, len(df) + 1))
        df['etl_loaded_at'] = datetime.now().isoformat(timespec='seconds')

        self.dw['Dim_Location_Context'] = df
        self.audit.step('Dim_Location_Context', 'transform', len(loc), len(df),
                        time.perf_counter() - t0,
                        note=f'SK su {nk}, {len(df)} paesi, tipi normalizzati')
        return df

    def build_fact_covid_trend(self):
        """Costruisce il Fact_Covid_Trend con FK lookup + validazione orfani.

        Pattern del corso (slide "Building the Fact Table - FK Lookup"):
          1. Read source
          2. Lookup location SK via merge su iso_code
          3. Lookup time SK via merge su date
          4. Validate: no orphan rows (NULL FKs dopo lookup)
          5. FIX CUMULATI: rendi monotoni i cumulati per paese (cummax)
          6. MISURE NORMALIZZATE pre-calcolate (per Tableau)
          7. Output: SK + measures + audit timestamp

        NATURA E UNITA' DELLE MISURE (cruciale per Tableau):
          --- FLUSSO giornaliero (additiva -> aggregazione SUM) ---
          new_cases, new_deaths   -> conteggi giornalieri (persone)
          new_tests               -> test giornalieri (conteggio)
          --- CUMULATA (NON additiva -> aggregazione MAX) ---
          total_cases, total_deaths    -> conteggi cumulati (persone)
          people_vaccinated            -> persone vaccinate, CONTEGGIO ASSOLUTO
          --- INDICE 0-100 (non additivo -> aggregazione AVG) ---
          stringency_index             -> indice 0-100
          --- NORMALIZZATE pre-calcolate (riga per riga) ---
          cases_per_million, deaths_per_million   -> per 1.000.000 ab. (usa MAX)
          tests_per_thousand                      -> per 1.000 ab. (usa MAX)
          vaccination_coverage_pct                -> % popolazione (0-100, usa MAX)
          cfr_pct                                 -> letalita' % = total_deaths/total_cases*100
        """
        t0 = time.perf_counter()

        # Seleziono solo le colonne dello schema da ciascuna tabella.
        ep  = self.src['Epidemic_Trend'][['iso_code', 'date',
              'new_cases', 'total_cases', 'new_deaths', 'total_deaths']]
        pre = self.src['Prevention_Measures'][['iso_code', 'date',
              'new_tests', 'stringency_index', 'people_vaccinated']]
        dim_loc  = self.dw['Dim_Location_Context']
        dim_time = self.dw['Dim_Time']

        # 1. JOIN epidemic + prevention sulla composite key (iso_code, date)
        src_fact = ep.merge(pre, on=['iso_code', 'date'], how='left')
        rows_in = len(src_fact)

        # 2. location_sk
        src_fact = src_fact.merge(
            dim_loc[['iso_code', 'location_sk']],
            on='iso_code', how='left'
        )

        # 3. time_sk
        src_fact['date'] = pd.to_datetime(src_fact['date'])
        src_fact = src_fact.merge(
            dim_time[['date', 'time_sk']],
            on='date', how='left'
        )

        # 4. Validate orphans
        fk_cols = ['location_sk', 'time_sk']
        orphans = src_fact[src_fact[fk_cols].isnull().any(axis=1)]
        if len(orphans) > 0:
            self.audit.step('Fact_Covid_Trend', 'transform', rows_in, 0,
                            time.perf_counter() - t0,
                            status='fail',
                            note=f'{len(orphans)} righe orfane su FK')
            raise ValueError(
                f'Validazione FK fallita: {len(orphans)} righe orfane.\n'
                f'Esempio:\n{orphans.head(3)}'
            )

        # 5.  cummax per paese garantisce monotonia e rende MAX un'aggregazione sicura in Tableau.
        src_fact = src_fact.sort_values(['location_sk', 'date'])
        cum_cols = ['total_cases', 'total_deaths', 'people_vaccinated']
        n_fixed = 0
        for c in cum_cols:
            before = src_fact[c].copy()
            src_fact[c] = src_fact.groupby('location_sk')[c].cummax()
            n_fixed += int((src_fact[c] != before).sum())

        # 5b. COERENZA TIPI: le misure numeriche come float, niente negativi
        #     sui flussi (un conteggio giornaliero non puo' essere negativo).
        flow_cols = ['new_cases', 'new_deaths', 'new_tests']
        for c in flow_cols:
            src_fact[c] = pd.to_numeric(src_fact[c], errors='coerce').clip(lower=0)

        # 6. MISURE NORMALIZZATE pre-calcolate (population dalla dim)
        src_fact = src_fact.merge(
            dim_loc[['location_sk', 'population']], on='location_sk', how='left'
        )
        pop = src_fact['population'].replace(0, np.nan)
        src_fact['cases_per_million']        = (src_fact['total_cases']  / pop * 1_000_000).round(2)
        src_fact['deaths_per_million']       = (src_fact['total_deaths'] / pop * 1_000_000).round(2)
        src_fact['tests_per_thousand']       = (src_fact['new_tests']    / pop * 1_000).round(4)
        src_fact['vaccination_coverage_pct'] = (src_fact['people_vaccinated'] / pop * 100).round(2)
        # CFR valido solo con casi sufficienti: nei primi giorni i decessi
        # possono precedere i casi registrati (lag) e gonfiano il rapporto.
        # Sotto i 100 casi cumulati -> NULL; il risultato e' comunque limitato a [0,100].
        cfr = src_fact['total_deaths'] / src_fact['total_cases'].replace(0, np.nan) * 100
        src_fact['cfr_pct'] = np.where(
            src_fact['total_cases'] >= 100,
            cfr.clip(lower=0, upper=100).round(3),
            np.nan
        )

        # 7. Output: SK + misure schema + misure normalizzate + audit
        all_measures = list(self.schema['measures'].keys())
        present_measures = [m for m in all_measures if m in src_fact.columns]
        derived = ['cases_per_million', 'deaths_per_million', 'tests_per_thousand',
                   'vaccination_coverage_pct', 'cfr_pct']

        fact_cols = fk_cols + present_measures + derived
        df = src_fact[fact_cols].copy()
        df['location_sk'] = df['location_sk'].astype(int)
        df['time_sk']     = df['time_sk'].astype(int)
        df.insert(0, 'fact_sk', range(1, len(df) + 1))
        df['etl_loaded_at'] = datetime.now().isoformat(timespec='seconds')

        self.dw['Fact_Covid_Trend'] = df
        self.audit.step('Fact_Covid_Trend', 'transform', rows_in, len(df),
                        time.perf_counter() - t0,
                        note=(f'FK lookup OK, {len(present_measures)} misure + '
                              f'{len(derived)} normalizzate, {n_fixed} valori cumulati corretti'))
        return df

    def load_csv(self, tables):
        """Esporta tutte le tabelle DW in CSV."""
        t0 = time.perf_counter()
        total = 0
        for name, path in tables.items():
            if name in self.dw:
                self.dw[name].to_csv(path, index=False)
                total += len(self.dw[name])
        self.audit.step('all tables', 'load', total, total,
                        time.perf_counter() - t0,
                        note=f'{len(tables)} CSV scritti')

print('ETLPipeline pronta')

ETLPipeline pronta


## 5. Extract — Reconciled DB
Le tre tabelle del Reconciled DB (output di L3) sono le sorgenti.

In [21]:
# EXTRACT: leggo le 3 tabelle pulite del Reconciled DB
pipeline = ETLPipeline(schema, audit)

pipeline.extract('Locations',          LOCATIONS_CLEAN)
pipeline.extract('Epidemic_Trend',     EPIDEMIC_CLEAN)
pipeline.extract('Prevention_Measures', PREVENTION_CLEAN)

print('\nEXTRACT completato')
for name, df in pipeline.src.items():
    print(f'   {name:<25} {len(df):>8,} righe × {len(df.columns)} colonne')


EXTRACT completato
   Locations                      171 righe × 9 colonne
   Epidemic_Trend             284,298 righe × 6 colonne
   Prevention_Measures        284,298 righe × 5 colonne


## 6. Transform - costruzione DW

**Le dimensioni prima del fatto**, perché il fatto ha bisogno delle SK per il lookup FK.

>  In questo step le tre tabelle vengono rese
> coerenti tra loro: tipi uniformi (numerici come `float`, chiavi/testi come `string`,
> SK come `int`), **cumulati resi monotoni** per paese (un cumulato non puo' decrescere),
> e **misure normalizzate pre-calcolate** nel fatto.
>
> **Natura delle misure** (per aggregazione corretta in Tableau):
> - *Flusso giornaliero* (additiva, **SUM**): `new_cases`, `new_deaths`, `new_tests`
> - *Cumulata* (non additiva, **MAX**): `total_cases`, `total_deaths`, `people_vaccinated` (conteggio assoluto di persone)
> - *Indice 0-100* (non additiva, **AVG**): `stringency_index`
> - *Normalizzate* pre-calcolate: `cases_per_million`, `deaths_per_million` (MAX), `tests_per_thousand` (MAX), `vaccination_coverage_pct` (MAX, 0-100), `cfr_pct`


In [22]:
# TRANSFORM: genero la dimensione tempo (Dim_Time) dal range di date
dim_time = pipeline.build_dim_time()
print(f'Dim_Time: {len(dim_time):,} righe')
dim_time.head(3)

Dim_Time: 1,688 righe


,time_sk,date,month,quarter,year,day_of_week,is_weekend,days_since_start,etl_loaded_at
0,1,2020-01-01,2020-01,2020Q1,2020,2,False,0,2026-06-14T09:18:19
1,2,2020-01-02,2020-01,2020Q1,2020,3,False,1,2026-06-14T09:18:19
2,3,2020-01-03,2020-01,2020Q1,2020,4,False,2,2026-06-14T09:18:19


In [23]:
# TRANSFORM: costruisco la dimensione luogo con chiave surrogata e tipi coerenti
dim_loc = pipeline.build_dim_location_context()
print(f'Dim_Location_Context: {len(dim_loc):,} righe')
dim_loc.head(3)

Dim_Location_Context: 171 righe


,location_sk,iso_code,location,continent,population,population_density,gdp_per_capita,median_age,hospital_beds_per_thousand,human_development_index,etl_loaded_at
0,1,AFG,Afghanistan,Asia,41128772.0,54.42,1803.99,18.6,0.50,0.51,2026-06-14T09:18:19
1,2,ALB,Albania,Europe,2842318.0,104.87,11803.43,38.0,2.89,0.80,2026-06-14T09:18:19
2,3,DZA,Algeria,Africa,44903228.0,17.35,13913.84,29.1,1.90,0.75,2026-06-14T09:18:19


In [24]:
# TRANSFORM: costruisco il fatto (lookup delle FK + controllo orfani + misure derivate)
fact = pipeline.build_fact_covid_trend()
print(f'Fact_Covid_Trend: {len(fact):,} righe')
fact.head(3)

Fact_Covid_Trend: 284,298 righe


,fact_sk,location_sk,time_sk,new_cases,total_cases,new_deaths,total_deaths,new_tests,stringency_index,people_vaccinated,cases_per_million,deaths_per_million,tests_per_thousand,vaccination_coverage_pct,cfr_pct,etl_loaded_at
0,1,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2026-06-14T09:18:20
1,2,1,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2026-06-14T09:18:20
2,3,1,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2026-06-14T09:18:20


##  7. Verifica integrità referenziale post-load

Ulteriore controllo: ogni `location_sk`/`time_sk`

In [25]:
# Controllo l'integrita referenziale: ogni FK del fatto deve esistere nelle dimensioni
def check_referential_integrity(fact, dim, fk_col, dim_pk_col):
    fact_keys = set(fact[fk_col].unique())
    dim_keys  = set(dim[dim_pk_col].unique())
    missing = fact_keys - dim_keys
    return len(missing), missing

n_miss_loc, _  = check_referential_integrity(fact, dim_loc,  'location_sk', 'location_sk')
n_miss_time, _ = check_referential_integrity(fact, dim_time, 'time_sk',     'time_sk')

print(f'FK location_sk: {n_miss_loc} chiavi orfane')
print(f'FK time_sk:     {n_miss_time} chiavi orfane')

assert n_miss_loc == 0 and n_miss_time == 0, 'Integrità referenziale violata'
print('\nIntegrità referenziale OK')

FK location_sk: 0 chiavi orfane
FK time_sk:     0 chiavi orfane

Integrità referenziale OK


##  8. Load — Export CSV per Tableau Public

In [26]:
# LOAD: salvo dimensioni, fatto e log di audit su CSV
pipeline.load_csv({
    'Dim_Time':             OUT_DIM_TIME,
    'Dim_Location_Context': OUT_DIM_LOC,
    'Fact_Covid_Trend':     OUT_FACT,
})

# Audit log su disco
audit.to_dataframe().to_csv(OUT_AUDIT, index=False)

print('Export completato:')
print(f'   {OUT_DIM_TIME}')
print(f'   {OUT_DIM_LOC}')
print(f'   {OUT_FACT}')
print(f'   {OUT_AUDIT}')

Export completato:
   /content/drive/MyDrive/Covid-19 Analysis/dw/Dim_Time.csv
   /content/drive/MyDrive/Covid-19 Analysis/dw/Dim_Location_Context.csv
   /content/drive/MyDrive/Covid-19 Analysis/dw/Fact_Covid_Trend.csv
   /content/drive/MyDrive/Covid-19 Analysis/dw/etl_log.csv


## 9. Aggiornamento `schema.json` con i conteggi reali


In [27]:
# Aggiorno schema.json con righe e percorsi effettivi prodotti dall'ETL
schema['etl_run'] = {
    'executed_at': datetime.now().isoformat(timespec='seconds'),
    'source':      'L5_Colab1_ETL_Covid.ipynb',
    'audit_log':   OUT_AUDIT,
    'dw_dir':      DW_DIR
}

schema['dimension_tables']['Dim_Time']['rows'] = len(dim_time)
schema['dimension_tables']['Dim_Time']['path'] = OUT_DIM_TIME

schema['dimension_tables']['Dim_Location_Context']['rows'] = len(dim_loc)
schema['dimension_tables']['Dim_Location_Context']['path'] = OUT_DIM_LOC

schema['fact_table']['rows'] = len(fact)
schema['fact_table']['path'] = OUT_FACT

with open(OUT_SCHEMA_UPDATED, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False, default=str)

print(f'schema.json aggiornato → {OUT_SCHEMA_UPDATED}')

schema.json aggiornato → /content/drive/MyDrive/Covid-19 Analysis/schema.json


## 10. Sample OLAP Query


In [28]:
# Prova OLAP: roll-up dei nuovi casi per continente e anno (verifica del round-trip dai CSV)
# Carica dai CSV (per verificare che il round-trip funzioni)
f = pd.read_csv(OUT_FACT)
dl = pd.read_csv(OUT_DIM_LOC)
dt = pd.read_csv(OUT_DIM_TIME)

# Roll-up: total new_cases per continent × year
sample = (f.merge(dl[['location_sk', 'continent']], on='location_sk')
           .merge(dt[['time_sk', 'year']], on='time_sk')
           .groupby(['continent', 'year'])['new_cases']
           .sum()
           .reset_index()
           .sort_values(['year', 'new_cases'], ascending=[True, False]))

print('Sample OLAP: nuovi casi per continente per anno\n')
print(sample.to_string(index=False))

Sample OLAP: nuovi casi per continente per anno

    continent  year   new_cases
         Asia  2020  17161077.0
South America  2020  12744552.0
       Europe  2020  12187158.0
North America  2020   3012622.0
       Africa  2020   1394886.0
      Oceania  2020     54719.0
         Asia  2021  50663657.0
       Europe  2021  32560105.0
South America  2021  25906056.0
North America  2021   7051782.0
       Africa  2021   3111885.0
      Oceania  2021    491766.0
       Europe  2022 116038746.0
         Asia  2022 115596795.0
South America  2022  26230738.0
      Oceania  2022  12578039.0
North America  2022   8682426.0
       Africa  2022   1563017.0
         Asia  2023  43559545.0
       Europe  2023   6043492.0
South America  2023   2071218.0
      Oceania  2023   1455212.0
North America  2023    834017.0
       Africa  2023     74184.0
       Europe  2024    427345.0
      Oceania  2024    344923.0
         Asia  2024    219123.0
South America  2024    181906.0
North America  2024    

## 🎯 11. Riepilogo


schema.json (L4)  →    fact_table , dim_tables , measures

ETLPipeline      →      extract , transform , load+audit

CSV per Tableau (L6)   →   Dim_Time.csv  , Dim_Location_Context.csv , Fact_Covid_Trend.csv. ,  etl_log.csv




---

### Aggiornamento coerenza dati (per Tableau)

- **Tipi uniformi** nelle 3 tabelle (float per le misure, string per testi/chiavi, int per le SK).
- **Cumulati monotoni**: `total_cases`, `total_deaths`, `people_vaccinated` corretti con `cummax` per paese.
- **Flussi non negativi**: `new_cases/new_deaths/new_tests` con `clip(lower=0)`.
- **Misure normalizzate** gia' pronte nel fatto: `cases_per_million`, `deaths_per_million`, `tests_per_thousand`, `vaccination_coverage_pct`, `cfr_pct`.
- I valori di contesto mancanti (GDP, posti letto, HDI, ...) restano `NULL`: Tableau li esclude automaticamente dalle viste, senza inventare dati.
